# SmartBite Date-Synth 1000 Colab Workflow

This notebook runs the same date-recognition fine-tuning flow in Google Colab:
1. Mount Drive
2. Unzip your `minidatesynth1000.zip` dataset
3. Validate PP-OCR dataset layout (`train_images/`, `val_images/`, labels)
4. Install Paddle + PaddleOCR training deps
5. Run training with PP-OCRv5 config


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!rm -rf /content/dataset
!mkdir -p /content/dataset

In [ ]:
!unzip -q "/content/drive/My Drive/sb-colab/ppocrv5_date_synth_dataset.zip" -d "/content/dataset/"

In [ ]:
from pathlib import Path
DATASET_ROOT = Path("/content/dataset/ppocrv5_date_synth_dataset")  # adjust exact folder


In [ ]:
print("DATASET_ROOT =", DATASET_ROOT)
print("train_label lines =", sum(1 for _ in open(DATASET_ROOT / "train_label.txt", "r", encoding="utf-8")))
print("val_label lines =", sum(1 for _ in open(DATASET_ROOT / "val_label.txt", "r", encoding="utf-8")))


DATASET_ROOT = /content/dataset/ppocrv5_date_synth_dataset
train_label lines = 115200
val_label lines = 12800


In [ ]:
from pathlib import Path

def find_dataset_root(base: Path) -> Path:
    direct = base
    if (direct / 'train_label.txt').exists() and (direct / 'val_label.txt').exists():
        return direct

    for candidate in sorted(base.rglob('*')):
        if not candidate.is_dir():
            continue
        if (candidate / 'train_label.txt').exists() and (candidate / 'val_label.txt').exists():
            return candidate

    raise FileNotFoundError('Could not find dataset root containing train_label.txt and val_label.txt')

BASE_UNZIP_DIR = Path('/content/dataset')
DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
print('DATASET_ROOT =', DATASET_ROOT)

for split in ('train', 'val'):
    label_file = DATASET_ROOT / f'{split}_label.txt'
    lines = [line for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    missing = 0
    for line in lines:
        rel_path = line.split('\t', 1)[0]
        if not (DATASET_ROOT / rel_path).exists():
            missing += 1
    print(f'{split}: labels={len(lines)} missing_files={missing}')


DATASET_ROOT = /content/dataset/ppocrv5_date_synth_dataset
train: labels=115200 missing_files=0
val: labels=12800 missing_files=0


## Wheelhouse (No Internet Installs in Colab)

This notebook installs dependencies from Drive-hosted wheels only.

Expected folders in Drive:
- `/content/drive/My Drive/wheels/colab-cu118` for GPU runtime
- `/content/drive/My Drive/wheels/colab-cpu` for CPU runtime

Build these wheel folders once on your machine, upload to Drive, then run this notebook.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Use existing pretrained model from Drive (no Colab-side download).
PRETRAINED_MODEL = Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams')
assert PRETRAINED_MODEL.exists(), f'Missing pretrained model: {PRETRAINED_MODEL}'
print('Found pretrained model:', PRETRAINED_MODEL)

# Use PaddleOCR repo archive from Drive instead of git clone.
PADDELOCR_ZIP = Path('/content/drive/My Drive/sb-colab/PaddleOCR.zip')
PADDELOCR_DIR = Path('/content/PaddleOCR')
if not PADDELOCR_DIR.exists():
    assert PADDELOCR_ZIP.exists(), f'Missing PaddleOCR zip: {PADDELOCR_ZIP}'
    !unzip -q "/content/drive/My Drive/sb-colab/PaddleOCR.zip" -d /content

%cd /content/PaddleOCR

# Avoid slow model-source connectivity checks during runtime.
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

has_gpu = os.system('nvidia-smi > /dev/null 2>&1') == 0
print('GPU available:', has_gpu)

wheel_zip = Path('/content/drive/My Drive/sb-colab/colab-cu118.zip' if has_gpu else '/content/drive/My Drive/sb-colab/colab-cpu.zip')
wheelhouse = Path('/content/wheels/colab-cu118' if has_gpu else '/content/wheels/colab-cpu')
if not wheelhouse.exists():
    assert wheel_zip.exists(), f'Missing wheelhouse zip: {wheel_zip}'
    wheelhouse.parent.mkdir(parents=True, exist_ok=True)
    !unzip -q "{wheel_zip}" -d "/content/wheels"

assert wheelhouse.exists(), f'Wheelhouse did not extract correctly: {wheelhouse}'
print('Using wheelhouse:', wheelhouse)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheelhouse), *args]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if has_gpu:
    pip_install(['paddlepaddle-gpu'])
else:
    pip_install(['paddlepaddle'])

pip_install(['-r', '/content/PaddleOCR/requirements.txt'])

import paddle
print('paddle version:', paddle.__version__)
print('compiled_with_cuda:', paddle.is_compiled_with_cuda())


Found pretrained model: /content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams
/content/PaddleOCR
GPU available: True
Using wheelhouse: /content/wheels/colab-cu118
>> /usr/bin/python3 -m pip install --no-index --find-links /content/wheels/colab-cu118 paddlepaddle-gpu
>> /usr/bin/python3 -m pip install --no-index --find-links /content/wheels/colab-cu118 -r /content/PaddleOCR/requirements.txt


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


paddle version: 3.3.1
compiled_with_cuda: True


In [ ]:
!python3 --version

Python 3.12.13


In [ ]:
from pathlib import Path
import yaml
import copy

# Paths
PRETRAINED_MODEL = Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams')
BASE_CONFIG = Path('/content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml')
OUTPUT_DIR = Path('/content/drive/My Drive/sb-colab/output/ppocrv5_date_synth_1000_run')
PATCHED_CONFIG = OUTPUT_DIR / 'patched_config_fast.yml'

# Training targets
EPOCHS = 20
BATCH_SIZE = 384          # Start here. If stable later, try 640 or 768.
LEARNING_RATE = 0.0005
TRAIN_IMAGE_SHAPE = [3, 48, 320]

# Speed mode
USE_MULTISCALE = False    # Critical for big speedup
TRAIN_NUM_WORKERS = 16
EVAL_NUM_WORKERS = 4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert BASE_CONFIG.exists(), f'Missing config: {BASE_CONFIG}'
assert PRETRAINED_MODEL.exists(), f'Missing pretrained model: {PRETRAINED_MODEL}'
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'
assert (DATASET_ROOT / 'train_label.txt').exists(), 'Missing train_label.txt'
assert (DATASET_ROOT / 'val_label.txt').exists(), 'Missing val_label.txt'

with open(BASE_CONFIG, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# ----------------------------
# Global
# ----------------------------
cfg['Global']['epoch_num'] = EPOCHS
cfg['Global']['pretrained_model'] = str(PRETRAINED_MODEL)
cfg['Global']['save_model_dir'] = str(OUTPUT_DIR)
cfg['Global']['d2s_train_image_shape'] = TRAIN_IMAGE_SHAPE

# Keep logs reasonable
cfg['Global']['print_batch_step'] = 100
cfg['Global']['eval_batch_step'] = [0, 900]   # roughly every ~2 epochs if fixed 512
cfg['Global']['save_epoch_step'] = 1

# ----------------------------
# Optimizer
# ----------------------------
cfg['Optimizer']['lr']['learning_rate'] = LEARNING_RATE

# ----------------------------
# Train dataset paths
# ----------------------------
cfg['Train']['dataset']['data_dir'] = str(DATASET_ROOT)
cfg['Train']['dataset']['label_file_list'] = [str(DATASET_ROOT / 'train_label.txt')]

# ----------------------------
# Eval dataset paths
# ----------------------------
cfg['Eval']['dataset']['data_dir'] = str(DATASET_ROOT)
cfg['Eval']['dataset']['label_file_list'] = [str(DATASET_ROOT / 'val_label.txt')]

# ----------------------------
# Fast mode: disable multiscale
# ----------------------------
if not USE_MULTISCALE:
    # Switch to fixed-size standard rec training
    cfg['Train']['dataset']['name'] = 'SimpleDataSet'

    # Remove multiscale-only dataset keys if present
    cfg['Train']['dataset'].pop('ds_width', None)
    cfg['Train']['dataset'].pop('ext_op_transform_idx', None)

    # Build fixed-size train transforms
    cfg['Train']['dataset']['transforms'] = [
        {
            'DecodeImage': {
                'img_mode': 'BGR',
                'channel_first': False
            }
        },
        {'RecAug': None},
        {
            'MultiLabelEncode': {
                'gtc_encode': 'NRTRLabelEncode'
            }
        },
        {
            'RecResizeImg': {
                'image_shape': TRAIN_IMAGE_SHAPE
            }
        },
        {
            'KeepKeys': {
                'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']
            }
        }
    ]

    # Remove sampler completely
    cfg['Train'].pop('sampler', None)

    # Use real fixed batch
    cfg['Train']['loader']['batch_size_per_card'] = BATCH_SIZE
    cfg['Train']['loader']['num_workers'] = TRAIN_NUM_WORKERS
    cfg['Train']['loader']['shuffle'] = True
    cfg['Train']['loader']['drop_last'] = True

else:
    # If you ever want multiscale back
    cfg['Train']['loader']['batch_size_per_card'] = BATCH_SIZE
    cfg['Train']['loader']['num_workers'] = TRAIN_NUM_WORKERS
    cfg['Train']['sampler']['first_bs'] = BATCH_SIZE
    cfg['Train']['sampler']['fix_bs'] = False

# ----------------------------
# Eval loader
# ----------------------------
cfg['Eval']['loader']['batch_size_per_card'] = BATCH_SIZE
cfg['Eval']['loader']['num_workers'] = EVAL_NUM_WORKERS
cfg['Eval']['loader']['shuffle'] = False
cfg['Eval']['loader']['drop_last'] = False

with open(PATCHED_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Using base config:', BASE_CONFIG)
print('Using patched config:', PATCHED_CONFIG)
print('Using pretrained model:', PRETRAINED_MODEL)
print('Output dir:', OUTPUT_DIR)
print('USE_MULTISCALE:', USE_MULTISCALE)
print('Train dataset name:', cfg['Train']['dataset']['name'])
print('Train batch_size_per_card:', cfg['Train']['loader']['batch_size_per_card'])
print('Eval batch_size_per_card:', cfg['Eval']['loader']['batch_size_per_card'])
print('Train num_workers:', cfg['Train']['loader']['num_workers'])
print('Eval num_workers:', cfg['Eval']['loader']['num_workers'])
if 'sampler' in cfg['Train']:
    print('Sampler still active:', cfg['Train']['sampler'])
else:
    print('Sampler removed: fixed-batch training is active')

Using base config: /content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml
Using patched config: /content/drive/My Drive/sb-colab/output/ppocrv5_date_synth_1000_run/patched_config_fast.yml
Using pretrained model: /content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams
Output dir: /content/drive/My Drive/sb-colab/output/ppocrv5_date_synth_1000_run
USE_MULTISCALE: False
Train dataset name: SimpleDataSet
Train batch_size_per_card: 384
Eval batch_size_per_card: 384
Train num_workers: 16
Eval num_workers: 4
Sampler removed: fixed-batch training is active


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    '/content/PaddleOCR/tools/train.py',
    '-c', str(PATCHED_CONFIG),
    '-o',
    'Global.use_gpu=True',
    'Global.print_batch_step=50',
    'Global.eval_batch_step=[0,1800]',
    'Global.save_epoch_step=5',
]

if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd = [c if c != 'Global.use_gpu=True' else 'Global.use_gpu=False' for c in cmd]

# Auto-resume from latest checkpoint if it exists.
latest_base = Path(OUTPUT_DIR) / 'latest'
latest_params = Path(str(latest_base) + '.pdparams')
if latest_params.exists():
    cmd.append(f'Global.checkpoints={latest_base}')
    print('Resuming from:', latest_base)

# 🔴 FIX 2: Stream logs in real-time instead of buffering them
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='') # Prints PaddleOCR logs to the cell immediately

process.wait()

if process.returncode == 0:
    print('\nTraining finished successfully.')
    print('Artifacts in:', OUTPUT_DIR)
else:
    print(f'\nTraining FAILED with exit code {process.returncode}.')


Resuming from: /content/drive/My Drive/sb-colab/output/ppocrv5_date_synth_1000_run/latest
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/03/28 07:29:33] ppocr INFO: Architecture : 
[2026/03/28 07:29:33] ppocr INFO:     Backbone : 
[2026/03/28 07:29:33] ppocr INFO:         name : PPHGNetV2_B4
[2026/03/28 07:29:33] ppocr INFO:         text_rec : True
[2026/03/28 07:29:33] ppocr INFO:     Head : 
[2026/03/28 07:29:33] ppocr INFO:         head_list : 
[2026/03/28 07:29:33] ppocr INFO:             CTCHead : 
[2026/03/28 07:29:33] ppocr INFO:                 Head : 
[2026/03/28 07:29:33] ppocr INFO:                     fc_decay : 1e-05
[2026/03/28 07:29:33] ppocr INFO:     

In [ ]:
from pathlib import Path

print("OUTPUT_DIR:", OUTPUT_DIR)
print("exists:", OUTPUT_DIR.exists())
print("files:", [p.name for p in OUTPUT_DIR.glob("*")][:20])

log_file = OUTPUT_DIR / "train.log"
print("train.log exists:", log_file.exists())
if log_file.exists():
    print(log_file.read_text(encoding="utf-8")[-4000:])


OUTPUT_DIR: /content/drive/My Drive/sb-colab/output/ppocrv5_date_synth_1000_run
exists: True
files: ['patched_config_fast.yml', 'config.yml', 'iter_epoch_5.pdopt', 'iter_epoch_5.pdparams', 'iter_epoch_5.states', 'best_model', 'best_accuracy.pdopt', 'best_accuracy.pdparams', 'best_accuracy.states', 'iter_epoch_10.pdopt', 'iter_epoch_10.pdparams', 'iter_epoch_10.states', 'latest.pdopt', 'latest.pdparams', 'latest.states', 'train.log', 'iter_epoch_15.pdopt', 'iter_epoch_15.pdparams', 'iter_epoch_15.states', 'iter_epoch_20.pdopt']
train.log exists: True
 global_step: 2250, lr: 0.000020, acc: 0.925781, norm_edit_dis: 0.987204, CTCLoss: 0.237499, NRTRLoss: 1.318112, loss: 1.555783, avg_reader_cost: 0.00224 s, avg_batch_cost: 2.08623 s, avg_samples: 384.0, ips: 184.06449 samples/s, eta: 0:16:03, max_mem_reserved: 52731 MB, max_mem_allocated: 49765 MB
[2026/03/28 08:54:16] ppocr INFO: epoch: [19/20], global_step: 2300, lr: 0.000017, acc: 0.919271, norm_edit_dis: 0.987651, CTCLoss: 0.217095, NR

In [ ]:
from pathlib import Path
import shutil

# Uses your existing variables exactly as you set them
# OUTPUT_DIR must already point to your training output folder.
# FINAL_MODEL_DRIVE_DIR should be set by you to your desired Drive destination.
# Example (only if you haven't set it yet):
FINAL_MODEL_DRIVE_DIR = Path("/content/drive/My Drive/sb-colab")

assert OUTPUT_DIR.exists(), f"Missing OUTPUT_DIR: {OUTPUT_DIR}"
assert FINAL_MODEL_DRIVE_DIR.parent.exists(), f"Missing parent dir: {FINAL_MODEL_DRIVE_DIR.parent}"

FINAL_MODEL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy full run artifacts (recommended for resume + audit)
for item in OUTPUT_DIR.iterdir():
    target = FINAL_MODEL_DRIVE_DIR / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

print("Saved model artifacts to:", FINAL_MODEL_DRIVE_DIR)


Saved model artifacts to: /content/drive/My Drive/sb-colab


## Optional: Full run

For full training, set:
- `EPOCHS = 30`
- keep `BATCH_SIZE = 32` (or lower if OOM)

Then rerun the last two cells.
